In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import shap

from pathlib import Path

pd.set_option("display.max_columns", None)
plt.style.use("ggplot")
sns.set_theme(style="whitegrid")

import os
os.makedirs("reports/figures", exist_ok=True)

In [ ]:
def find_file(filename, extra_dirs=()):
    candidates = [Path(filename)]
    for d in list(extra_dirs) + ["data/processed", "../data/processed", "../../data/processed",
                                  "models", "../models", "../../models"]:
        candidates.append(Path(d) / filename)
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError(
        f"Could not find '{filename}'. Looked in: {[str(c) for c in candidates]}"
    )

model_path = None
for name in ["boosting_tuned_best_model.pkl", "baseline_best_model.pkl"]:
    try:
        model_path = find_file(name)
        break
    except FileNotFoundError:
        continue

if model_path is None:
    raise FileNotFoundError(
        "No saved model found. Run notebook 03 or 04 first and make sure "
        "the .pkl file is next to this notebook or in models/."
    )

pipeline = joblib.load(model_path)
print(f"Loaded model: {model_path}")

train_df = pd.read_csv(find_file("train_features.csv"))
test_df = pd.read_csv(find_file("test_features.csv"))
print("train_df:", train_df.shape, " test_df:", test_df.shape)

In [ ]:
categorical_cols = ["main_category", "currency", "country", "launch_weekday"]
numeric_cols = [
    "usd_goal_real", "goal_log", "campaign_duration",
    "launch_month", "launch_quarter",
    "category_frequency", "country_success_rate",
]
feature_cols = categorical_cols + numeric_cols

X_test = test_df[feature_cols]
y_test = test_df["target"]

preprocessor = pipeline.named_steps["preprocessor"]
model = pipeline.named_steps["model"]

X_test_transformed = preprocessor.transform(X_test)
if hasattr(X_test_transformed, "toarray"):  # handle sparse OneHotEncoder output
    X_test_transformed = X_test_transformed.toarray()

feature_names = preprocessor.get_feature_names_out()
X_test_transformed_df = pd.DataFrame(X_test_transformed, columns=feature_names, index=X_test.index)

print("Transformed feature matrix:", X_test_transformed_df.shape)

# Day 7 — SHAP Explainability


### Sample for speed


In [ ]:
SAMPLE_SIZE = min(500, len(X_test_transformed_df))
sample_idx = X_test_transformed_df.sample(SAMPLE_SIZE, random_state=42).index

X_shap = X_test_transformed_df.loc[sample_idx]
print(f"Explaining {len(X_shap)} rows")

### Build the explainer and compute SHAP values

In [ ]:
explainer = shap.TreeExplainer(model)
shap_values = explainer(X_shap)

print("SHAP values shape:", shap_values.values.shape)


### Global importance — bar plot


In [ ]:
shap.plots.bar(shap_values, show=False, max_display=15)
plt.tight_layout()
plt.savefig("reports/figures/shap_global_importance.png", dpi=300, bbox_inches="tight")
plt.show()

### Summary plot (beeswarm)


In [ ]:
shap.plots.beeswarm(shap_values, show=False, max_display=15)
plt.tight_layout()
plt.savefig("reports/figures/shap_summary_beeswarm.png", dpi=300, bbox_inches="tight")
plt.show()

### Individual explanation — waterfall plot


In [ ]:
example_position = 0
shap.plots.waterfall(shap_values[example_position], show=False, max_display=12)
plt.tight_layout()
plt.savefig("reports/figures/shap_waterfall_example.png", dpi=300, bbox_inches="tight")
plt.show()

original_row = X_test.loc[[sample_idx[example_position]]]
actual_outcome = y_test.loc[sample_idx[example_position]]
predicted_proba = pipeline.predict_proba(original_row)[0, 1]

print(f"Actual outcome: {'successful' if actual_outcome == 1 else 'failed'}")
print(f"Predicted probability of success: {predicted_proba:.3f}")
original_row

### Day 7 deliverable — save the explanation for a few example campaigns

In [ ]:
top_features = pd.DataFrame({
    "feature": feature_names,
    "mean_abs_shap": np.abs(shap_values.values).mean(axis=0),
}).sort_values("mean_abs_shap", ascending=False)

top_features.to_csv("reports/shap_global_feature_importance.csv", index=False)
top_features.head(15)


# Day 8 — Counterfactual Engine


In [ ]:
def simulate_counterfactual(pipeline, base_row: pd.DataFrame, changes: dict) -> pd.DataFrame:
    """Apply `changes` (column -> new value) to a copy of `base_row` and compare
    predicted success probability before vs after."""
    modified_row = base_row.copy()
    for col, value in changes.items():
        modified_row[col] = value

    original_proba = pipeline.predict_proba(base_row)[0, 1]
    new_proba = pipeline.predict_proba(modified_row)[0, 1]

    return pd.DataFrame({
        "Scenario": ["Current", "What-if"],
        "Probability of Success": [original_proba, new_proba],
    }), modified_row


### Pick a campaign to run scenarios on

In [ ]:
base_row = X_test.loc[[sample_idx[example_position]]]
base_proba = pipeline.predict_proba(base_row)[0, 1]

print("Base campaign:")
display(base_row)
print(f"Current predicted probability of success: {base_proba:.3f}")


### Scenario A — Lower the funding goal

In [ ]:
new_goal = base_row["usd_goal_real"].values[0] * 0.5

result_a, _ = simulate_counterfactual(pipeline, base_row, {
    "usd_goal_real": new_goal,
    "goal_log": np.log1p(new_goal),
})
result_a["Change"] = f"Goal: ${base_row['usd_goal_real'].values[0]:,.0f} -> ${new_goal:,.0f}"
result_a

### Scenario B — Longer campaign duration

In [ ]:
new_duration = base_row["campaign_duration"].values[0] + 20

result_b, _ = simulate_counterfactual(pipeline, base_row, {
    "campaign_duration": new_duration,
})
result_b["Change"] = f"Duration: {base_row['campaign_duration'].values[0]:.0f}d -> {new_duration:.0f}d"
result_b

### Scenario C — Different category


In [ ]:
category_success = train_df.groupby("main_category")["target"].mean().sort_values(ascending=False)
best_category = category_success.index[0]

result_c, _ = simulate_counterfactual(pipeline, base_row, {
    "main_category": best_category,
})
result_c["Change"] = f"Category: {base_row['main_category'].values[0]} -> {best_category}"
result_c

### All scenarios side by side

In [ ]:
all_scenarios = pd.concat([
    result_a.assign(Scenario_Group="Lower Goal"),
    result_b.assign(Scenario_Group="Longer Duration"),
    result_c.assign(Scenario_Group="Best Category"),
])

fig, ax = plt.subplots(figsize=(10, 6))
for group in all_scenarios["Scenario_Group"].unique():
    subset = all_scenarios[all_scenarios["Scenario_Group"] == group]
    ax.plot(subset["Scenario"], subset["Probability of Success"], marker="o", label=group)

ax.axhline(base_proba, color="gray", linestyle="--", alpha=0.5, label="Baseline")
ax.set_ylabel("Predicted Probability of Success")
ax.set_title("Counterfactual Scenarios")
ax.legend()
plt.tight_layout()
plt.savefig("reports/figures/counterfactual_scenarios.png", dpi=300, bbox_inches="tight")
plt.show()

all_scenarios[["Scenario_Group", "Change", "Scenario", "Probability of Success"]]


### Save Day 8 deliverable

In [ ]:
all_scenarios.to_csv("reports/counterfactual_scenarios.csv", index=False)

import shutil
shutil.make_archive("shap_counterfactual_outputs", "zip", "reports")

try:
    from google.colab import files
    files.download("shap_counterfactual_outputs.zip")
except ImportError:
    print("Not running in Colab -- saved locally: shap_counterfactual_outputs.zip")
